# Solutions 2 — Asyncio, Threads, and Reliability

Six worked reference patterns covering structured task ownership, cancellation, synchronization, and bounded retries.

**Level:** Advanced · **Study mode:** Worked reference examples · **Prerequisites:** Relevant lessons and exercise workbook.

These solutions show one defensible approach each; multiple implementations may be correct. Review the contracts, test boundaries, and discuss design trade-offs before comparing line-by-line.

## 1. TaskGroup results in input order

Tasks are explicitly owned by the TaskGroup and joined before their results are accessed. This prevents orphan background tasks.

In [ ]:
import asyncio

async def double(value):
    await asyncio.sleep(0)
    return value * 2

async def run_all(values):
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(double(x)) for x in values]
    return [task.result() for task in tasks]

assert asyncio.run(run_all([1, 3, 5])) == [2, 6, 10]

**Extension:** What error policy would you use if one child fails?

## 2. Cancellation-safe cleanup

A finally block records cleanup even when an async task is cancelled. Cancellation should ordinarily be re-raised by the task.

In [ ]:
import asyncio

async def work(events):
    try:
        events.append("started")
        await asyncio.sleep(100)
    finally:
        events.append("closed")

async def run_cancel():
    events = []
    task = asyncio.create_task(work(events))
    await asyncio.sleep(0)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        events.append("cancelled")
    return events

assert asyncio.run(run_cancel()) == ["started", "closed", "cancelled"]

**Extension:** What would cleanup do if a socket also needed to be closed?

## 3. Bounded concurrency

A semaphore bounds simultaneously active coroutine sections. It is distinct from limiting a service's requests per second.

In [ ]:
import asyncio

async def bounded(values, limit=2):
    sem = asyncio.Semaphore(limit)
    active, peak = 0, 0
    async def one(value):
        nonlocal active, peak
        async with sem:
            active += 1
            peak = max(peak, active)
            await asyncio.sleep(0)
            active -= 1
            return value ** 2
    results = await asyncio.gather(*(one(n) for n in values))
    return results, peak

results, peak = asyncio.run(bounded(range(5)))
assert results == [0, 1, 4, 9, 16] and peak <= 2

**Extension:** How would you add a per-second token bucket?

## 4. Synchronized thread counter

A Lock protects a shared read-modify-write sequence. Tests should assert the final invariant, not the unpredictable scheduling order.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

counter, lock = [0], Lock()
def update(times):
    for _ in range(times):
        with lock:
            counter[0] += 1

with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(update, [50] * 4))
assert counter[0] == 200

**Extension:** Can locking become a bottleneck with high contention?

## 5. Retry with fake sleeps

Retry only a narrow transient error class and stop after a fixed maximum number of attempts. Inject sleep so tests remain instant.

In [ ]:
class TransientError(Exception): pass

def retry(call, attempts=4, sleep=lambda seconds: None):
    if attempts < 1:
        raise ValueError("attempts must be positive")
    for index in range(attempts):
        try:
            return call()
        except TransientError:
            if index + 1 == attempts:
                raise
            sleep(0.1 * 2 ** index)

state = {"calls": 0}
def flaky():
    state["calls"] += 1
    if state["calls"] < 3:
        raise TransientError()
    return "ok"

delays = []
assert retry(flaky, sleep=delays.append) == "ok"
assert delays == [0.1, 0.2]

**Extension:** How do you avoid two thousand clients retrying at the same instant?

## 6. Idempotency with key mapping

The in-memory example shows the logical contract; durable atomic records are required for correctness across processes or crashes.

In [ ]:
class ActionStore:
    def __init__(self):
        self.records = {}
        self.number = 0
    def execute(self, key, payload):
        if key in self.records:
            saved_payload, result = self.records[key]
            if saved_payload != payload:
                raise ValueError("conflicting idempotency key")
            return result
        self.number += 1
        self.records[key] = (payload, self.number)
        return self.number

store = ActionStore()
assert store.execute("k1", "charge:10") == store.execute("k1", "charge:10")
assert store.number == 1

**Extension:** How would a unique database constraint and transaction protect concurrent requests?

## Recap

- Reference solutions illustrate testing and contracts rather than exhaustive answers.
- Explain one alternative design for each pattern.
- Re-run every code cell in a fresh kernel before copying into a project.

**Next:** Work independently on the unsolved questions in the corresponding workbook.